# tps-meter — Colab runner

The code comes from the zip package on Drive; **the work runs on Colab's local disk** (fast), and every finished shard is **backed up to Drive** immediately. If the session disconnects, re-running cells 1–2 and the generation cell is enough: it continues where it stopped.

**Drive layout** (`My Drive/tps-meter/`):
- `tps-meter-code.zip` — code package built on the computer with `python scripts/pack_for_colab.py`
- `data/synthetic/v1/` — generated data and backup shards (written by the notebook)
- `runs/` — training outputs (Phase 5)

**Runtime:** data generation uses the CPU only and is limited by memory bandwidth; choose a runtime with many cores (*Runtime → Change runtime type*). Cell 2 prints the core count, cell 4 measures the best number of workers.

In [ ]:
# 1) Drive + code + paths
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, shutil, zipfile
DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/tps-meter')
WORK = pathlib.Path('/content/tps-meter')
if WORK.exists():
    shutil.rmtree(WORK)
zipfile.ZipFile(DRIVE_DIR / 'tps-meter-code.zip').extractall(WORK)

LOCAL_DATA = pathlib.Path('/content/data')              # fast local disk
MIRROR = DRIVE_DIR / 'data' / 'synthetic' / 'v1'          # persistent backup
LOCAL_DATA.mkdir(exist_ok=True)
MIRROR.mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'runs').mkdir(parents=True, exist_ok=True)
os.environ['TPS_DATA_ROOT'] = str(LOCAL_DATA)
os.environ['TPS_RUNS_ROOT'] = str(DRIVE_DIR / 'runs')
os.chdir(WORK)
print('code:', WORK, '| local data:', LOCAL_DATA, '| Drive backup:', MIRROR)
print('in Drive backup:', sorted(p.name for p in MIRROR.iterdir()))

In [ ]:
# 2) Installation and hardware
!pip install -q -e ".[dev]"
# The running kernel does not see a package installed in the same session; add it to the path.
import os, sys
sys.path.insert(0, str(WORK / 'src'))
import torch
print('CPU cores:', os.cpu_count())
!free -g | head -2
from tpslib.runtime import get_device, describe_device
print('torch', torch.__version__, '|', describe_device(get_device()))

In [ ]:
# 3) Tests (about 1–2 min). Do not start generation unless all of them pass.
!python -m pytest -q

In [ ]:
# 4) Throughput test: finds the fastest number of workers and estimates the full generation time (a few minutes).
!python scripts/bench_generation.py

In [ ]:
# 5) FULL GENERATION. Set WORKERS to the value suggested by cell 4.
# If the session drops: re-run 1, 2 and this cell; finished shards are restored from Drive.
WORKERS = os.cpu_count()
!python scripts/generate_dataset.py --splits train val test test_interp test_faulty --workers {WORKERS} --mirror "{MIRROR}"

In [ ]:
# 6) Check: are the files on Drive complete?
import h5py, json
for name in ['train', 'val', 'test', 'test_interp', 'test_faulty']:
    f = MIRROR / f'{name}.h5'
    if not f.exists():
        print(f'{name:12s} MISSING'); continue
    with h5py.File(f, 'r') as h:
        st = json.loads(h.attrs['stats'])
        print(f"{name:12s} n={h['dT'].shape[0]:6d} ok={st['n_ok']:6d} failed={st['n_failed']} "
              f"mean sim={st['sim_time_mean_s']:.1f}s  {f.stat().st_size/1e6:.0f} MB")
print(json.dumps(json.loads((MIRROR / 'manifest.json').read_text())['splits'], indent=1)[:2000])

## Phase 5 — M2 training

**Runtime:** GPU (L4 is enough; A100 also works). The training data are re-noised on the CPU every epoch, so the core count matters too (L4: 12).

If the session drops: re-run cells 1, 2, 7 and the training cell you were in **with the same command**; runs continue from the last epoch under `runs/` on Drive.

In order: 8 (5-epoch trial) → 9 (data size study) → 10 (5-seed ensemble) → 11 (ablation).

In [ ]:
# 7) Copy the data from Drive to the fast local disk (training reads from here)
dst = LOCAL_DATA / 'synthetic' / 'v1'
dst.mkdir(parents=True, exist_ok=True)
for name in ['train', 'val', 'test', 'test_interp', 'test_faulty']:
    src = MIRROR / f'{name}.h5'
    if src.exists() and not (dst / src.name).exists():
        shutil.copy2(src, dst / src.name)
shutil.copy2(MIRROR / 'manifest.json', dst / 'manifest.json')
print(sorted(p.name for p in dst.iterdir()))
NW = os.cpu_count()

In [ ]:
# 8) Short trial: one seed, 5 epochs (check of time and learning curve)
!python scripts/train_m2.py --seeds 1 --max-epochs 5 --tag trial --num-workers {NW}

In [ ]:
# 9) Data size study: with 10k and 25k samples (50k = seed1 in cell 10). Answers whether more data is needed.
!python scripts/train_m2.py --seeds 1 --max-train 10000 --tag n10k --num-workers {NW}
!python scripts/train_m2.py --seeds 1 --max-train 25000 --tag n25k --num-workers {NW}

In [ ]:
# 9b) Diagnosis: which samples the error comes from (val only; the test sets stay locked). ~2 min
!python scripts/diagnose.py --run n10k_m2_seed1 --num-workers {NW}


In [ ]:
# 9c) Comparison (one seed, 50k): (a) all data, (b) contact asymmetry <= 10. ~40 min each.
# Then the diagnosis of both. When done, send the last two tables.
!python scripts/train_m2.py --seeds 1 --tag v1 --num-workers {NW}
!python scripts/train_m2.py --seeds 1 --tag v1sym --max-asym 10 --num-workers {NW}
!python scripts/diagnose.py --run v1_m2_seed1 --num-workers {NW}
!python scripts/diagnose.py --run v1sym_m2_seed1 --num-workers {NW}


In [ ]:
# 9d) Finish model (a): early stopping no longer acts before epoch 30.
# v1_m2_seed1 continues from epoch 28 to 60 (~23 min), then the diagnosis.
!python scripts/train_m2.py --seeds 1 --tag v1 --num-workers {NW}
!python scripts/diagnose.py --run v1_m2_seed1 --num-workers {NW}


In [ ]:
# 10) Full training: 5-seed ensemble, realistic contact (asymmetry <= 10; seed1 = v1sym from 9c)
!python scripts/train_m2.py --seeds 1 2 3 4 5 --tag v1sym --max-asym 10 --num-workers {NW}


In [ ]:
# 11) Ablation: model with the k = 1.5–2.5 W/m·K band removed from train/val (one seed, asymmetry <= 10)
!python scripts/train_m2.py --seeds 1 --tag ablation --max-asym 10 --exclude-k-band 1.5 2.5 --num-workers {NW}


In [ ]:
# 12) Summary: best epoch of every run
import json, glob
for f in sorted(glob.glob(str(DRIVE_DIR / 'runs' / '*' / 'summary.json'))):
    s = json.load(open(f)); b = s['best']
    print(f"{pathlib.Path(f).parent.name:28s} ep={s['best_epoch']:3d}/{s['epochs_run']:3d} "
          f"k med={100*b['val_med_rel_err_ln_k']:.2f}% p95={100*b['val_p95_rel_err_ln_k']:.2f}% "
          f"a med={100*b['val_med_rel_err_ln_alpha']:.2f}% cov95(k)={b['val_cov95_ln_k']:.2f}")

## Phase 6 — Evaluation, compact model, ONNX

The test sets are used **once** here. Outputs on Drive: `tps-meter/reports/phase6/` and `tps-meter/phase6_bundle.zip`.

In [ ]:
# 13) Compact device model (~62k parameters, 2 members). Independent of the ensemble training; can also run in parallel in a separate session. ~1 h
!python scripts/train_m2.py --config configs/train_m2_compact.yaml --seeds 1 2 --tag v1c --max-asym 10 --num-workers {NW}

In [ ]:
# 14) Final evaluation (ensemble + ablation + compact must be ready). ~20–30 min
!python scripts/evaluate.py --ensemble-tag v1sym --seeds 1 2 3 4 5 --ablation ablation_m2_seed1 --compact-tag v1c --compact-seeds 1 2 --max-asym 10 --num-workers {NW}

## Phase 6 re-run with English figures + trust checks v2

The model is not retrained. The full evaluation repeats the Phase 6 numbers (same models, same seeds) with English figures. Thresholds and the variance scale of trust v2 are chosen on **val**; this is the **second use** of the test sets. Outputs: `tps-meter/reports/phase6/`, `tps-meter/reports/phase6_trust/` and the bundle zips. The cell runs on its own (cells 1–2 are not needed).


In [ ]:
# 15) Phase 6 evaluation (English figures) + trust checks v2 and variance scale (SECOND USE OF THE TEST SETS).
# Runs on its own; ~30-40 min.
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, shutil, zipfile
DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/tps-meter')
WORK = pathlib.Path('/content/tps-meter')
shutil.rmtree(WORK, ignore_errors=True)
zipfile.ZipFile(DRIVE_DIR / 'tps-meter-code.zip').extractall(WORK)
os.chdir(WORK)
dst = pathlib.Path('/content/data/synthetic/v1'); dst.mkdir(parents=True, exist_ok=True)
for f in ['val.h5', 'test.h5', 'test_interp.h5', 'test_faulty.h5', 'manifest.json']:
    if not (dst / f).exists():
        shutil.copy2(DRIVE_DIR / 'data' / 'synthetic' / 'v1' / f, dst / f)
os.environ['TPS_DATA_ROOT'] = '/content/data'
os.environ['TPS_RUNS_ROOT'] = str(DRIVE_DIR / 'runs')
!pip install -q -e ".[dev]"
!python -m pytest -q tests/test_trust.py tests/test_phase6.py
!python scripts/evaluate.py --ensemble-tag v1sym --seeds 1 2 3 4 5 --ablation ablation_m2_seed1 --compact-tag v1c --compact-seeds 1 2 --max-asym 10 --num-workers {os.cpu_count()}
!python scripts/evaluate.py --trust-only --ensemble-tag v1sym --seeds 1 2 3 4 5 --max-asym 10 --num-workers {os.cpu_count()}
print((DRIVE_DIR / 'reports' / 'phase6_trust' / 'TRUST_REPORT.md').read_text(encoding='utf-8'))